# 04 - CSD Analysis (SECONDARY)

Explore Critical Slowing Down results from `data/results/csd/`.  
Reproduces Figures 5-7: volcano plots, example protein time series,
composite score distributions, and temporal specificity analysis.  
**CSD is the secondary analysis** that provides mechanistic support for DNB findings.
This notebook reads pre-computed results only.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

results_dir = os.path.join('..', config['paths']['results'], 'csd')
print('CSD results directory:', results_dir)
print('Files available:', os.listdir(results_dir) if os.path.exists(results_dir) else 'DIRECTORY NOT FOUND')

## Figure 1A - Volcano Plot: Kendall Tau for Variance

Each point is one protein. X-axis: Kendall tau (converters vs. stable MCI difference).  
Y-axis: -log10(FDR-corrected p-value). Significant proteins are highlighted.

In [ ]:
# Load protein-level CSD statistics
tau_df = pd.read_csv(os.path.join(results_dir, 'kendall_tau_variance.csv'))

print(f'Total proteins tested: {len(tau_df)}')
print(f'Significant at FDR < 0.05: {(tau_df["fdr_pvalue"] < 0.05).sum()}')

# Volcano plot
fig, ax = plt.subplots(figsize=(8, 6))
neg_log_p = -np.log10(tau_df['fdr_pvalue'].clip(lower=1e-300))
sig_mask = tau_df['fdr_pvalue'] < 0.05

ax.scatter(tau_df.loc[~sig_mask, 'effect_size'], neg_log_p[~sig_mask],
           c='grey', alpha=0.3, s=8, label='Not significant')
ax.scatter(tau_df.loc[sig_mask, 'effect_size'], neg_log_p[sig_mask],
           c='firebrick', alpha=0.6, s=12, label=f'FDR < 0.05 (n={sig_mask.sum()})')

ax.axhline(-np.log10(0.05), color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('Effect Size (rank-biserial r)')
ax.set_ylabel('-log10(FDR p-value)')
ax.set_title('CSD Variance Trend: Converters vs. Stable MCI')
ax.legend()
plt.tight_layout()
plt.show()

## Example Protein Time Series

Show protein level and rolling-window variance/AR1 for a top CSD-significant protein
in an example converter participant.

In [ ]:
# Load per-participant CSD results
csd_all = pd.read_csv(os.path.join(results_dir, 'csd_all_proteins_participants.csv'))

# Pick the top protein by effect size
top_protein = tau_df.sort_values('effect_size', ascending=False).iloc[0]['protein']
print(f'Top CSD protein: {top_protein}')

# Filter to converters for this protein
conv_data = csd_all[(csd_all['protein'] == top_protein) & (csd_all['trajectory'] == config['adni']['converter_group'])]
print(f'Converter participants with this protein: {conv_data["RID"].nunique()}')

# Show summary statistics
print(f'Mean Kendall tau (variance) in converters: {conv_data["tau_variance"].mean():.3f}')
print(f'Mean Kendall tau (AR1) in converters: {conv_data["tau_ar1"].mean():.3f}')

## Composite CSD Score Distribution

Distribution of composite CSD scores for converters vs. stable MCI.  
Higher scores indicate stronger critical slowing down signal.

In [ ]:
# Load composite scores
composite = pd.read_csv(os.path.join(results_dir, 'composite_csd_scores.csv'))

converters = composite[composite['TRAJECTORY'] == config['adni']['converter_group']]
stable = composite[composite['TRAJECTORY'] == config['adni']['stable_group']]

# Wilcoxon rank-sum test
stat, pval = stats.mannwhitneyu(converters['composite_csd_score'], stable['composite_csd_score'],
                                 alternative='greater')

fig, ax = plt.subplots(figsize=(7, 5))
parts = ax.violinplot([stable['composite_csd_score'].dropna(), converters['composite_csd_score'].dropna()],
                      positions=[0, 1], showmedians=True)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Stable MCI', 'MCI Converters'])
ax.set_ylabel('Composite CSD Score')
ax.set_title(f'Composite CSD Score by Group (p = {pval:.2e})')
plt.tight_layout()
plt.show()

print(f'Converters: mean={converters["composite_csd_score"].mean():.3f}, n={len(converters)}')
print(f'Stable MCI: mean={stable["composite_csd_score"].mean():.3f}, n={len(stable)}')

## Temporal Specificity Analysis

Composite CSD score by time-to-conversion strata.  
CSD theory predicts a peak at 12-24 months before conversion (not monotonic increase).

In [ ]:
# Load temporal specificity results
temporal = pd.read_csv(os.path.join(results_dir, 'temporal_specificity.csv'))

fig, ax = plt.subplots(figsize=(8, 5))
strata_order = ['>=36m', '24-36m', '12-24m', '<12m']
temporal['time_stratum'] = pd.Categorical(temporal['time_stratum'], categories=strata_order, ordered=True)
temporal = temporal.sort_values('time_stratum')

ax.bar(range(len(temporal)), temporal['mean_csd_score'], yerr=temporal['sem_csd_score'],
       color='steelblue', capsize=4, edgecolor='white')
ax.set_xticks(range(len(temporal)))
ax.set_xticklabels(temporal['time_stratum'])
ax.set_xlabel('Time to Conversion')
ax.set_ylabel('Mean Composite CSD Score')
ax.set_title('CSD Score by Time-to-Conversion Window')
plt.tight_layout()
plt.show()

# Kruskal-Wallis test if raw data available
print('Temporal specificity summary:')
print(temporal[['time_stratum', 'mean_csd_score', 'sem_csd_score', 'n']].to_string(index=False))